In [17]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import cv2

#EXCLUDE_DATASETS = {'Deepfakes','DeepFakeDetection'}

class FFPPFrameDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, compression='c23', use_fft=False, use_dct=False, transform=None):
        print("Dataset 초기화 중…")
        self.transform = transform
        self.samples = []
        roots = [os.path.join(root_dir, 'original_sequences'),
                 os.path.join(root_dir, 'manipulated_sequences')]
        for label, base in enumerate(roots):
            for method in os.listdir(base):
                # 제외할 데이터셋 스킵
                # if base.endswith('manipulated_sequences') and method in EXCLUDE_DATASETS:
                #     continue
                full_dir = os.path.join(base, method, compression, 'mtcnn')
                if not os.path.isdir(full_dir):
                    continue
                for subdir, _, files in os.walk(full_dir):
                    for fname in files:
                        if fname.lower().endswith(('png','jpg','jpeg')):
                            self.samples.append((os.path.join(subdir, fname), label))
        print(f"총 샘플 수: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        arr = np.array(img)[:, :, ::-1].astype(np.float32) / 255.0  # RGB→BGR and scale
        channels = [arr.transpose(2, 0, 1)]
        if self.use_fft:
            fft_map = extract_fft(arr)
            channels.append(fft_map.transpose(2, 0, 1))
        if self.use_dct:
            dct_map = extract_dct(arr)
            channels.append(dct_map[None])
        x = np.concatenate(channels, axis=0)
        return torch.from_numpy(x), torch.tensor(label, dtype=torch.long)
    
# 데이터셋 준비
transform = transforms.Resize((224,224))
dataset = FFPPFrameDataset(
        root_dir="/home/oem/deepfake/hdd_5TB/FF++",
        transform=transform
    )

# train/val split
total_len = len(dataset)
train_size = int(0.8 * total_len)
val_size   = total_len - train_size
print(f"Train/Val split: {train_size}/{val_size}")
train_ds, val_ds = random_split(dataset, [train_size, val_size])

# DataLoader
print("\n▶ DataLoader 생성 중…")
train_loader = DataLoader(
        train_ds,
        batch_size=16,
        shuffle=True,
        num_workers=4,
        pin_memory=False
    )
val_loader = DataLoader(
        val_ds,
        batch_size=16,
        shuffle=False,
        num_workers=4,
        pin_memory=False
    )
print("DataLoader 준비 완료.\n")

Dataset 초기화 중…
총 샘플 수: 840405
Train/Val split: 672324/168081

▶ DataLoader 생성 중…
DataLoader 준비 완료.



In [18]:
from collections import Counter

# 전체 샘플에서 레이블만 추출
labels = [label for _, label in dataset.samples]

# 클래스별 카운트
cnt = Counter(labels)
total = len(labels)

print("=== 클래스 분포 ===")
for cls in sorted(cnt):
    print(f"Label {cls}: {cnt[cls]} samples ({cnt[cls]/total*100:.2f}%)")

=== 클래스 분포 ===
Label 0: 197461 samples (23.50%)
Label 1: 642944 samples (76.50%)


In [ ]:
import numpy as np
from collections import Counter
import torch
from torch.utils.data import TensorDataset, DataLoader
from imblearn.over_sampling import BorderlineSMOTE
from tqdm import tqdm

# 0) train_ds 는 FFPPFrameDataset 의 train_split (이미 transform 포함)  
#    model.forward_features(x) 로 feature 뽑아냄  
model.eval()
feats, labels = [], []
with torch.no_grad():
    for x, y in tqdm(train_loader, desc="Extracting features"):
        x = x.to(device)
        f = model.forward_features(x)         # (B, D)  ex) D=768
        feats.append(f.cpu().numpy())
        labels.append(y.numpy())
feats = np.vstack(feats)    # (N, D)
labels = np.concatenate(labels)  # (N,)

print("원본 class 분포:", Counter(labels))

# 1) Borderline-SMOTE 로 클래스 0을 클래스 1 개수에 맞춰 합성
target_num = int((labels == 1).sum())
smote = BorderlineSMOTE(
    sampling_strategy={0: target_num},
    random_state=42,
    kind='borderline-1'  # 또는 'borderline-2'
)
X_res, y_res = smote.fit_resample(feats, labels)
print("SMOTE 후 class 분포:", Counter(y_res))

# 2) TensorDataset & DataLoader 로 되돌려서 학습에 사용
X_res_t = torch.from_numpy(X_res).float()
y_res_t = torch.from_numpy(y_res).long()
bal_train_ds = TensorDataset(X_res_t, y_res_t)
bal_train_loader = DataLoader(
    bal_train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True
)

# 이제 bal_train_loader 로 classifier 학습을 진행하세요.


NameError: name 'model' is not defined

In [2]:
from PIL import Image
import cv2
import numpy as np

def save_fft_dct_images(input_path, fft_output_path, dct_output_path):
    # 1) RGB 이미지 로드
    img = Image.open(input_path).convert('RGB')
    arr = np.array(img)  # H×W×3, RGB

    # 2) BGR 변환 (OpenCV는 BGR 사용)
    bgr = arr[:, :, ::-1]

    # 3) FFT 추출
    channels = []
    for ch in cv2.split(bgr):
        # DFT → shift → magnitude → 로그 스케일
        dft   = cv2.dft(np.float32(ch), flags=cv2.DFT_COMPLEX_OUTPUT)
        shift = np.fft.fftshift(dft)
        mag   = cv2.magnitude(shift[:,:,0], shift[:,:,1])
        log_mag = 20 * np.log(mag + 1)
        channels.append(log_mag)
    fft_map = np.stack(channels, axis=2)  # H×W×3

    # 정규화 후 저장
    fft_norm = cv2.normalize(fft_map, None, alpha=0, beta=255,
                             norm_type=cv2.NORM_MINMAX)
    fft_img = Image.fromarray(fft_norm.astype(np.uint8))
    fft_img.save(fft_output_path)

    # 4) DCT 추출 (그레이스케일)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    gray_f = np.float32(gray) / 255.0
    dct_map = cv2.dct(gray_f)

    # 정규화 후 저장
    dct_norm = cv2.normalize(dct_map, None, alpha=0, beta=255,
                             norm_type=cv2.NORM_MINMAX)
    dct_img = Image.fromarray(dct_norm.astype(np.uint8))
    dct_img.save(dct_output_path)

# 사용 예시
save_fft_dct_images(
    input_path="/home/oem/deepfake/hdd_5TB/FF++/original_sequences/actors/c23/mtcnn/01__exit_phone_room/0028.png",
    fft_output_path="fft_output.png",
    dct_output_path="dct_output.png"
)


In [2]:
from datasets import load_dataset

ds = load_dataset("xingjunm/WildDeepfake")

/home/oem/anaconda3/envs/py38/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATA_PATH = '/home/oem/deepfake/hdd_5TB/WildDeepfake' #현재 폴더 위치
ds.save_to_disk(DATA_PATH)

Saving the dataset (20/20 shards): 100%|██████████| 165662/165662 [06:07<00:00, 451.21 examples/s] 


In [5]:
from datasets import load_from_disk   # 또는 load_dataset
from pathlib import Path
from PIL import Image
import io

# arrow 파일이 들어 있는 디렉터리 경로
DATA_DIR = "/home/oem/deepfake/hdd_5TB/WildDeepfake"  # arrow 파일이 있는 폴더를 가리킵니다.

# 로컬에 캐시되어 있는 arrow dataset 을 불러옵니다.
dataset = load_from_disk(DATA_DIR)  # arrow 파일(.arrow)이 있는 폴더를 가리킵니다.
# 만약 load_dataset 을 쓰셨다면
# dataset = load_dataset("your_dataset_name", split="test")


In [8]:
# 사용 가능한 split 확인
print(dataset.keys())           # 예: dict_keys(['train', 'test'])

# train split 의 컬럼 이름과 첫 샘플 확인
print(dataset['train'].column_names)
print(dataset['train'][0])


dict_keys(['train', 'test'])
['__key__', '__url__', 'png']
{'__key__': './1/fake/131/1057', '__url__': '/home/oem/.cache/huggingface/hub/datasets--xingjunm--WildDeepfake/snapshots/f3835aaf281dd9f8d79b51c4e02f050d3f7af0b4/deepfake_in_the_wild/fake_train/1.tar.gz', 'png': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=224x224 at 0x7FA469E47AC0>}


In [9]:
from pathlib import Path
from tqdm import tqdm

# 1) HuggingFace에서 로드한 DatasetDict
#    dataset = load_dataset(...)

OUTPUT_ROOT = Path("/home/oem/deepfake/hdd_5TB/WildDeepfake_png")

for split in ["train", "test"]:
    ds = dataset[split]
    print(f"{split} split, samples: {len(ds)}")
    
    for ex in tqdm(ds, desc=f"Saving {split}", ncols=80):
        img: Image.Image = ex["png"]  # 이미 PIL.Image
        key: str         = ex["__key__"]  # e.g. "./1/fake/131/1057"
        
        # 2) key에서 선행 "./" 제거, 확장자 붙이기
        rel = key.lstrip("./") + ".png"  
        
        out_path = OUTPUT_ROOT / split / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # 3) 저장
        img.save(out_path, format="PNG")

print("모두 완료되었습니다.")


train split, samples: 1014437


Saving train: 100%|█████████████████| 1014437/1014437 [4:39:31<00:00, 60.48it/s]


test split, samples: 165662


Saving test: 100%|██████████████████████| 165662/165662 [48:57<00:00, 56.39it/s]

모두 완료되었습니다.


In [1]:
import torch

ckpt_path = "/home/oem/deepfake/Ourmethod/RGBsparial_step1/checkpoints/hornet_focal/hornet_ep001.pth"
ckpt      = torch.load(ckpt_path, map_location="cpu")   # GPU 없어도 확인만 할 땐 cpu

# 1) 최상위 키 확인
print("top-level keys :", ckpt.keys())
# 예) dict_keys(['model_state', 'optim_state', 'epoch', 'best_f1'])

# 2) 모델 state_dict 안의 파라미터 이름 몇 개만 미리 보기
state = ckpt['model_state']
print("num of params :", len(state))
for i, (k, v) in enumerate(state.items()):
    if i == 20: break                           # 앞 20개만
    print(f"{k:45s}  shape={tuple(v.shape)}")


top-level keys : odict_keys(['downsample_layers.0.0.weight', 'downsample_layers.0.0.bias', 'downsample_layers.0.1.weight', 'downsample_layers.0.1.bias', 'downsample_layers.1.0.weight', 'downsample_layers.1.0.bias', 'downsample_layers.1.1.weight', 'downsample_layers.1.1.bias', 'downsample_layers.2.0.weight', 'downsample_layers.2.0.bias', 'downsample_layers.2.1.weight', 'downsample_layers.2.1.bias', 'downsample_layers.3.0.weight', 'downsample_layers.3.0.bias', 'downsample_layers.3.1.weight', 'downsample_layers.3.1.bias', 'stages.0.0.gamma1', 'stages.0.0.gamma2', 'stages.0.0.norm1.weight', 'stages.0.0.norm1.bias', 'stages.0.0.gnconv.proj_in.weight', 'stages.0.0.gnconv.proj_in.bias', 'stages.0.0.gnconv.dwconv.weight', 'stages.0.0.gnconv.dwconv.bias', 'stages.0.0.gnconv.proj_out.weight', 'stages.0.0.gnconv.proj_out.bias', 'stages.0.0.gnconv.pws.0.weight', 'stages.0.0.gnconv.pws.0.bias', 'stages.0.0.norm2.weight', 'stages.0.0.norm2.bias', 'stages.0.0.pwconv1.weight', 'stages.0.0.pwconv1.bias

KeyError: 'model_state'